In [ ]:
#Decision Tree
import numpy as np
from collections import Counter

# Dataset
data = np.array([
    [1, 1, 0, 0, 1],  # T T F F T
    [0, 0, 1, 1, 1],  # F F T T T
    [1, 1, 1, 0, 1],  # T T T F T
    [1, 1, 0, 1, 1],  # T T F T T
    [0, 1, 1, 1, 0],  # F T T T F
    [0, 0, 1, 0, 0],  # F F T F F
    [1, 0, 0, 1, 0],  # T F F T F
    [1, 0, 1, 0, 0],  # T F T F F
])

X = data[:, :4]
y = data[:, 4]

feature_names = ['Home Match', 'Positive Balance', 'Bad Weather', 'Win Previous']

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None

class CustomDecisionTree:
    def __init__(self):
        self.root = None

    def fit(self, X, y):
        # Build the custom tree with the exact structure you specified
        self.root = self._build_custom_tree(X, y)

    def _build_custom_tree(self, X, y):
        # Root node: Positive Balance (index 1)
        root = Node(feature=1, threshold=0.5)  # Binary feature

        # Build left branch (Positive Balance = 0)
        mask_pb0 = X[:, 1] == 0
        X_pb0, y_pb0 = X[mask_pb0], y[mask_pb0]

        # Win Previous node for Positive Balance = 0
        win_prev_node = Node(feature=3, threshold=0.5)

        # Left branch of Win Previous (Win Previous = 0)
        mask_wp0 = X_pb0[:, 3] == 0
        y_wp0 = y_pb0[mask_wp0]
        leaf_wp0 = Node(value=0)  # Leaf: 0

        # Right branch of Win Previous (Win Previous = 1)
        mask_wp1 = X_pb0[:, 3] == 1
        X_wp1, y_wp1 = X_pb0[mask_wp1], y_pb0[mask_wp1]

        # Bad Weather node for Win Previous = 1
        bad_weather_node = Node(feature=2, threshold=0.5)

        # Left branch of Bad Weather (Bad Weather = 0)
        mask_bw0 = X_wp1[:, 2] == 0
        y_bw0 = y_wp1[mask_bw0]
        leaf_bw0 = Node(value=0)  # Leaf: 0

        # Right branch of Bad Weather (Bad Weather = 1)
        mask_bw1 = X_wp1[:, 2] == 1
        y_bw1 = y_wp1[mask_bw1]
        leaf_bw1 = Node(value=1)  # Leaf: 1

        # Connect Bad Weather node
        bad_weather_node.left = leaf_bw0
        bad_weather_node.right = leaf_bw1

        # Connect Win Previous node
        win_prev_node.left = leaf_wp0
        win_prev_node.right = bad_weather_node

        # Build right branch (Positive Balance = 1)
        mask_pb1 = X[:, 1] == 1
        X_pb1, y_pb1 = X[mask_pb1], y[mask_pb1]

        # Home Match node for Positive Balance = 1
        home_match_node = Node(feature=0, threshold=0.5)

        # Left branch of Home Match (Home Match = 0)
        mask_hm0 = X_pb1[:, 0] == 0
        y_hm0 = y_pb1[mask_hm0]
        leaf_hm0 = Node(value=0)  # Leaf: 0

        # Right branch of Home Match (Home Match = 1)
        mask_hm1 = X_pb1[:, 0] == 1
        y_hm1 = y_pb1[mask_hm1]
        leaf_hm1 = Node(value=1)  # Leaf: 1

        # Connect Home Match node
        home_match_node.left = leaf_hm0
        home_match_node.right = leaf_hm1

        # Connect root node
        root.left = win_prev_node   # Positive Balance = 0
        root.right = home_match_node  # Positive Balance = 1

        return root

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

    def print_tree(self, node=None, indent=""):
        if node is None:
            node = self.root

        if node.is_leaf():
            print(f"{indent}Leaf: {node.value}")
        else:
            feature_name = feature_names[node.feature]
            print(f"{indent}{feature_name}")

            # Left branch (value 0)
            print(f"{indent}  Value 0:")
            if node.left and not node.left.is_leaf():
                self.print_tree(node.left, indent + "    ")
            elif node.left:
                print(f"{indent}    Leaf: {node.left.value}")

            # Right branch (value 1)
            print(f"{indent}  Value 1:")
            if node.right and not node.right.is_leaf():
                self.print_tree(node.right, indent + "    ")
            elif node.right:
                print(f"{indent}    Leaf: {node.right.value}")

# Create and use the custom tree
custom_tree = CustomDecisionTree()
custom_tree.fit(X, y)

print("Custom Decision Tree Structure:")
custom_tree.print_tree()


Custom Decision Tree Structure:
Positive Balance
  Value 0:
    Win Previous
      Value 0:
        Leaf: 0
      Value 1:
        Bad Weather
          Value 0:
            Leaf: 0
          Value 1:
            Leaf: 1
  Value 1:
    Home Match
      Value 0:
        Leaf: 0
      Value 1:
        Leaf: 1


##Random forest

In [ ]:
import numpy as np
from collections import Counter

# Dataset
data = np.array([
    [1, 1, 0, 0, 1],  # T T F F T
    [0, 0, 1, 1, 1],  # F F T T T
    [1, 1, 1, 0, 1],  # T T T F T
    [1, 1, 0, 1, 1],  # T T F T T
    [0, 1, 1, 1, 0],  # F T T T F
    [0, 0, 1, 0, 0],  # F F T F F
    [1, 0, 0, 1, 0],  # T F F T F
    [1, 0, 1, 0, 0],  # T F T F F
])

# Split into features and outcomes
features = data[:, :4]
outcomes = data[:, 4]

feature_labels = ['Home Match', 'Positive Balance', 'Bad Weather', 'Win Previous']


class DecisionNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None


class MatchPredictor:
    def __init__(self):
        self.root = None

    def train(self, X, y):
        self.root = self._create_tree(X, y)

    def _create_tree(self, X, y):
        # Start with the most important factor: team's financial balance
        root = DecisionNode(feature=1, threshold=0.5)

        # When balance is negative
        negative_balance = X[:, 1] == 0
        X_neg, y_neg = X[negative_balance], y[negative_balance]

        # Check if they won last match
        check_prev_win = DecisionNode(feature=3, threshold=0.5)

        # No previous win -> likely loss
        no_prev_win = X_neg[:, 3] == 0
        y_no_win = y_neg[no_prev_win]
        loss_leaf = DecisionNode(value=0)

        # Had previous win -> check weather
        had_prev_win = X_neg[:, 3] == 1
        X_prev_win, y_prev_win = X_neg[had_prev_win], y_neg[had_prev_win]

        check_weather = DecisionNode(feature=2, threshold=0.5)

        # Good weather -> loss
        good_weather = X_prev_win[:, 2] == 0
        y_good = y_prev_win[good_weather]
        loss_in_good = DecisionNode(value=0)

        # Bad weather -> win
        bad_weather = X_prev_win[:, 2] == 1
        y_bad = y_prev_win[bad_weather]
        win_in_bad = DecisionNode(value=1)

        check_weather.left = loss_in_good
        check_weather.right = win_in_bad

        check_prev_win.left = loss_leaf
        check_prev_win.right = check_weather

        # When balance is positive
        positive_balance = X[:, 1] == 1
        X_pos, y_pos = X[positive_balance], y[positive_balance]

        # Check if playing at home
        check_home = DecisionNode(feature=0, threshold=0.5)

        # Away match -> loss
        away_match = X_pos[:, 0] == 0
        y_away = y_pos[away_match]
        away_loss = DecisionNode(value=0)

        # Home match -> win
        home_match = X_pos[:, 0] == 1
        y_home = y_pos[home_match]
        home_win = DecisionNode(value=1)

        check_home.left = away_loss
        check_home.right = home_win

        # Connect everything
        root.left = check_prev_win
        root.right = check_home

        return root

    def predict(self, X):
        return np.array([self._make_prediction(match, self.root) for match in X])

    def _make_prediction(self, match, node):
        if node.is_leaf():
            return node.value

        if match[node.feature] <= node.threshold:
            return self._make_prediction(match, node.left)
        return self._make_prediction(match, node.right)

    def show_structure(self, node=None, indent=""):
        if node is None:
            node = self.root

        if node.is_leaf():
            result = "WIN" if node.value == 1 else "LOSS"
            print(f"{indent}→ Predict: {result}")
        else:
            feature = feature_labels[node.feature]
            print(f"{indent}Check: {feature}")

            print(f"{indent}  If NO:")
            if node.left and not node.left.is_leaf():
                self.show_structure(node.left, indent + "    ")
            elif node.left:
                result = "WIN" if node.left.value == 1 else "LOSS"
                print(f"{indent}    → Predict: {result}")

            print(f"{indent}  If YES:")
            if node.right and not node.right.is_leaf():
                self.show_structure(node.right, indent + "    ")
            elif node.right:
                result = "WIN" if node.right.value == 1 else "LOSS"
                print(f"{indent}    → Predict: {result}")


class EnsemblePredictor:
    def __init__(self, num_predictors=3):
        self.num_predictors = num_predictors
        self.predictors = []

    def train(self, X, y):
        self.predictors = []

        for i in range(self.num_predictors):
            predictor = MatchPredictor()
            predictor.train(X, y)
            self.predictors.append(predictor)

            print(f"\n--- Predictor {i+1} Decision Rules ---")
            predictor.show_structure()

    def _majority_vote(self, predictions):
        counts = Counter(predictions)
        return counts.most_common(1)[0][0]

    def predict(self, X):
        all_predictions = np.array([p.predict(X) for p in self.predictors])
        votes = np.swapaxes(all_predictions, 0, 1)
        final_predictions = np.array([self._majority_vote(v) for v in votes])
        return final_predictions


def calculate_accuracy(actual, predicted):
    correct = np.sum(actual == predicted)
    total = len(actual)
    return correct / total


# Train and test the ensemble
print("FOOTBALL MATCH OUTCOME PREDICTOR")
print("=" * 60)
print("\nTraining ensemble of predictors...\n")

ensemble = EnsemblePredictor(num_predictors=3)
ensemble.train(features, outcomes)

# Make predictions
predictions = ensemble.predict(features)
accuracy = calculate_accuracy(outcomes, predictions)

print(f"\n{'=' * 60}")
print(f"Overall Accuracy: {accuracy:.1%}")
print(f"{'=' * 60}\n")

print("Match-by-Match Results:")
print("-" * 60)

for i in range(len(features)):
    actual = "WIN" if outcomes[i] == 1 else "LOSS"
    predicted = "WIN" if predictions[i] == 1 else "LOSS"

    conditions = [feature_labels[j] for j in range(4) if features[i,j] == 1]
    status = "✓" if outcomes[i] == predictions[i] else "✗"

    print(f"Match {i+1}: {', '.join(conditions)}")
    print(f"  Actual: {actual} | Predicted: {predicted} {status}\n")

FOOTBALL MATCH OUTCOME PREDICTOR

Training ensemble of predictors...


--- Predictor 1 Decision Rules ---
Check: Positive Balance
  If NO:
    Check: Win Previous
      If NO:
        → Predict: LOSS
      If YES:
        Check: Bad Weather
          If NO:
            → Predict: LOSS
          If YES:
            → Predict: WIN
  If YES:
    Check: Home Match
      If NO:
        → Predict: LOSS
      If YES:
        → Predict: WIN

--- Predictor 2 Decision Rules ---
Check: Positive Balance
  If NO:
    Check: Win Previous
      If NO:
        → Predict: LOSS
      If YES:
        Check: Bad Weather
          If NO:
            → Predict: LOSS
          If YES:
            → Predict: WIN
  If YES:
    Check: Home Match
      If NO:
        → Predict: LOSS
      If YES:
        → Predict: WIN

--- Predictor 3 Decision Rules ---
Check: Positive Balance
  If NO:
    Check: Win Previous
      If NO:
        → Predict: LOSS
      If YES:
        Check: Bad Weather
          If NO:
        

In [ ]:
#Bagging and bootstrapping

import numpy as np
from collections import Counter

# Dataset
data = np.array([
    [1, 1, 0, 0, 1],  # T T F F T
    [0, 0, 1, 1, 1],  # F F T T T
    [1, 1, 1, 0, 1],  # T T T F T
    [1, 1, 0, 1, 1],  # T T F T T
    [0, 1, 1, 1, 0],  # F T T T F
    [0, 0, 1, 0, 0],  # F F T F F
    [1, 0, 0, 1, 0],  # T F F T F
    [1, 0, 1, 0, 0],  # T F T F F
])

X = data[:, :4]
y = data[:, 4]

feature_names = ['Home Match', 'Positive Balance', 'Bad Weather', 'Win Previous']

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None

class CustomDecisionTree:
    def __init__(self):
        self.root = None

    def fit(self, X, y):
        # Build the custom tree with the exact structure you specified
        self.root = self._build_custom_tree(X, y)

    def _build_custom_tree(self, X, y):
        # Root node: Positive Balance (index 1)
        root = Node(feature=1, threshold=0.5)  # Binary feature

        # Build left branch (Positive Balance = 0)
        mask_pb0 = X[:, 1] == 0
        X_pb0, y_pb0 = X[mask_pb0], y[mask_pb0]

        # Win Previous node for Positive Balance = 0
        win_prev_node = Node(feature=3, threshold=0.5)

        # Left branch of Win Previous (Win Previous = 0)
        mask_wp0 = X_pb0[:, 3] == 0
        y_wp0 = y_pb0[mask_wp0]
        leaf_wp0 = Node(value=0)  # Leaf: 0

        # Right branch of Win Previous (Win Previous = 1)
        mask_wp1 = X_pb0[:, 3] == 1
        X_wp1, y_wp1 = X_pb0[mask_wp1], y_pb0[mask_wp1]

        # Bad Weather node for Win Previous = 1
        bad_weather_node = Node(feature=2, threshold=0.5)

        # Left branch of Bad Weather (Bad Weather = 0)
        mask_bw0 = X_wp1[:, 2] == 0
        y_bw0 = y_wp1[mask_bw0]
        leaf_bw0 = Node(value=0)  # Leaf: 0

        # Right branch of Bad Weather (Bad Weather = 1)
        mask_bw1 = X_wp1[:, 2] == 1
        y_bw1 = y_wp1[mask_bw1]
        leaf_bw1 = Node(value=1)  # Leaf: 1

        # Connect Bad Weather node
        bad_weather_node.left = leaf_bw0
        bad_weather_node.right = leaf_bw1

        # Connect Win Previous node
        win_prev_node.left = leaf_wp0
        win_prev_node.right = bad_weather_node

        # Build right branch (Positive Balance = 1)
        mask_pb1 = X[:, 1] == 1
        X_pb1, y_pb1 = X[mask_pb1], y[mask_pb1]

        # Home Match node for Positive Balance = 1
        home_match_node = Node(feature=0, threshold=0.5)

        # Left branch of Home Match (Home Match = 0)
        mask_hm0 = X_pb1[:, 0] == 0
        y_hm0 = y_pb1[mask_hm0]
        leaf_hm0 = Node(value=0)  # Leaf: 0

        # Right branch of Home Match (Home Match = 1)
        mask_hm1 = X_pb1[:, 0] == 1
        y_hm1 = y_pb1[mask_hm1]
        leaf_hm1 = Node(value=1)  # Leaf: 1

        # Connect Home Match node
        home_match_node.left = leaf_hm0
        home_match_node.right = leaf_hm1

        # Connect root node
        root.left = win_prev_node   # Positive Balance = 0
        root.right = home_match_node  # Positive Balance = 1

        return root

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

    def print_tree(self, node=None, indent=""):
        if node is None:
            node = self.root

        if node.is_leaf():
            print(f"{indent}Leaf: {node.value}")
        else:
            feature_name = feature_names[node.feature]
            print(f"{indent}{feature_name}")

            # Left branch (value 0)
            print(f"{indent}  Value 0:")
            if node.left and not node.left.is_leaf():
                self.print_tree(node.left, indent + "    ")
            elif node.left:
                print(f"{indent}    Leaf: {node.left.value}")

            # Right branch (value 1)
            print(f"{indent}  Value 1:")
            if node.right and not node.right.is_leaf():
                self.print_tree(node.right, indent + "    ")
            elif node.right:
                print(f"{indent}    Leaf: {node.right.value}")


class BaggingClassifier:
    def __init__(self, n_estimators=3, random_state=42):
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.trees = []
        self.bootstrap_indices = []

    def fit(self, X, y):
        np.random.seed(self.random_state)
        n_samples = X.shape[0]

        self.trees = []
        self.bootstrap_indices = []

        for i in range(self.n_estimators):
            # Create bootstrap sample (sampling with replacement)
            bootstrap_idx = np.random.choice(n_samples, size=n_samples, replace=True)
            self.bootstrap_indices.append(bootstrap_idx)

            X_bootstrap = X[bootstrap_idx]
            y_bootstrap = y[bootstrap_idx]

            # Train a tree on this bootstrap sample
            tree = CustomDecisionTree()
            tree.fit(X_bootstrap, y_bootstrap)
            self.trees.append(tree)

            print(f"\n{'='*60}")
            print(f"Tree {i+1} - Bootstrap Sample Indices: {bootstrap_idx}")
            print(f"{'='*60}")
            tree.print_tree()

    def predict(self, X):
        # Get predictions from all trees
        predictions = np.array([tree.predict(X) for tree in self.trees])

        # Transpose to get predictions for each sample
        predictions = predictions.T

        # Majority voting for each sample
        final_predictions = []
        for sample_preds in predictions:
            counter = Counter(sample_preds)
            most_common = counter.most_common(1)[0][0]
            final_predictions.append(most_common)

        return np.array(final_predictions)


def accuracy_score(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)


# Train bagging classifier
print("\n" + "="*60)
print("BAGGING ENSEMBLE WITH DECISION TREES")
print("="*60)

bagging = BaggingClassifier(n_estimators=3, random_state=42)
bagging.fit(X, y)

# Make predictions
y_pred = bagging.predict(X)

# Calculate accuracy
acc = accuracy_score(y, y_pred)

print("\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"Overall Accuracy: {acc:.2%}")
print("\nDetailed Predictions:")
print("-" * 60)

for i in range(len(X)):
    actual = y[i]
    predicted = y_pred[i]
    features = [feature_names[j] for j in range(4) if X[i,j] == 1]
    status = '✓' if actual == predicted else '✗'
    print(f"Instance {i+1}: {', '.join(features)}")
    print(f"  Actual: {actual}, Predicted: {predicted} {status}")
    print()


BAGGING ENSEMBLE WITH DECISION TREES

Tree 1 - Bootstrap Sample Indices: [6 3 4 6 2 7 4 4]
Positive Balance
  Value 0:
    Win Previous
      Value 0:
        Leaf: 0
      Value 1:
        Bad Weather
          Value 0:
            Leaf: 0
          Value 1:
            Leaf: 1
  Value 1:
    Home Match
      Value 0:
        Leaf: 0
      Value 1:
        Leaf: 1

Tree 2 - Bootstrap Sample Indices: [6 1 2 6 2 2 7 4]
Positive Balance
  Value 0:
    Win Previous
      Value 0:
        Leaf: 0
      Value 1:
        Bad Weather
          Value 0:
            Leaf: 0
          Value 1:
            Leaf: 1
  Value 1:
    Home Match
      Value 0:
        Leaf: 0
      Value 1:
        Leaf: 1

Tree 3 - Bootstrap Sample Indices: [3 7 7 2 5 4 1 7]
Positive Balance
  Value 0:
    Win Previous
      Value 0:
        Leaf: 0
      Value 1:
        Bad Weather
          Value 0:
            Leaf: 0
          Value 1:
            Leaf: 1
  Value 1:
    Home Match
      Value 0:
        Leaf: 0


# Proof: Why $$Var\left[\frac{1}{B_{\text{sum}}B_{\text{attribute}}}\sum_{i,j}x_{i,j}\right] < Var(x_{i,j})$$

# Let $X$ be the population random variable with $Var(X) = \sigma^2$.

# For $B$ groups with $m$ observations per group ($x_{i,j}$ i.i.d.):
# $Var\left[ \frac{1}{B} \sum_{j=1}^B \left( \frac{1}{m} \sum_{i=1}^m x_{i,j} \right) \right] < Var[x_{i,j}] < Var[X]$

# Which simplifies to:
# $\frac{\sigma^2}{Bm} < \sigma^2 = \sigma^2$

# The correct inequality is:
# $Var\left[ \frac{1}{B} \sum_{j=1}^B \left( \frac{1}{m} \sum_{i=1}^m x_{i,j} \right) \right] < Var\left[ \frac{1}{m} \sum_{i=1}^m x_{i,j} \right] < Var[x_{i,j}]$

# Or equivalently:
# $\frac{\sigma^2}{Bm} < \frac{\sigma^2}{m} < \sigma^2$

# Let $x_{i,j}$ be random variables with identical variance $Var(x_{i,j}) = \sigma^2.$

# Define the average:
# $\bar{x} = \frac{1}{B_{\text{sum}}B_{\text{attribute}}} \sum_{i=1}^{B_{\text{sum}}}\sum_{j=1}^{B_{\text{attribute}}} x_{i,j}.$

# Case 1: Independent (i.i.d.) random variables
# If all $x_{i,j}$ are independent, then
# $Var(\bar{x}) = \frac{1}{(B_{\text{sum}}B_{\text{attribute}})^2} \sum_{i=1}^{B_{\text{sum}}}\sum_{j=1}^{B_{\text{attribute}}}Var(x_{i,j}) = \frac{\sigma^2}{B_{\text{sum}}B_{\text{attribute}}}.$

# Since $B_{\text{sum}}B_{\text{attribute}} > 1$, we have
# $Var(\bar{x}) = \frac{\sigma^2}{B_{\text{sum}}B_{\text{attribute}}} < \sigma^2 = Var(x_{i,j}).$

# Hence,
# $\boxed{Var\left(\tfrac{1}{B_{\text{sum}}B_{\text{attribute}}}\sum_{i,j}x_{i,j}\right) < Var(x_{i,j}).}$

# Case 2: Weighted-average viewpoint
# For a general weighted average $Y = \sum_k w_k X_k$ with $\sum_k w_k = 1$ and uncorrelated variables $X_k$,
# $Var(Y) = \sigma^2 \sum_k w_k^2.$

# Because $\sum_k w_k^2 \le (\sum_k w_k)^2 = 1$ with strict inequality if more than one $w_k$ is non-zero,
# it follows that $Var(Y) < \sigma^2.$

# If $x_{i,j}$ are correlated,
# $Var(\bar{x}) = \frac{1}{n^2}\sum_{k,\ell}Cov(x_k,x_\ell) = \frac{\sigma^2}{n}\big(1+(n-1)\rho\big),$
# where $\rho$ is the average correlation.

# Thus $Var(\bar{x}) < \sigma^2$ whenever $\rho < 1$,
# and equality holds only when all samples are perfectly correlated.

# Conclusion
# Averaging independent or partially independent random variables reduces variance:
# $\boxed{Var\left[ \frac{1}{B_{\text{sum}}B_{\text{attribute}}} \sum_{i,j} x_{i,j} \right] < Var(x_{i,j})}.$